# B2-024 — Session 5: Scientific Inverse Problems

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

A forward problem predicts measurements from causes; an inverse problem recovers causes from measurements.
Here the cause is a single point source on a plate, at position $(x, y)\in[0.1,0.9]^2$ with strength $q$, and the measurements are field magnitudes at eight sensors.
This session builds the forward operator, shows why inversion is ill-posed, linearizes the problem on a grid and regularizes it with Tikhonov (ridge) regression, then trains a neural network on simulated pairs to invert directly, checking it with a physics residual and a coverage test.
The linear algebra is Book 1's `book1:matrix-multiplication`, `book1:invertibility-via-rank`, `book1:sum-of-squares-gradients`, `book1:linear-regression`, and `book1:l2-regularization`.

## 1. The forward operator

**Geometry.** Eight sensors sit on the boundary of the unit square, at the corners and edge midpoints:
$(0,0), (0.5,0), (1,0), (1,0.5), (1,1), (0.5,1), (0,1), (0,0.5)$ (`capstone_data.SENSORS_XY`).
They are mounted at height $h=0.25$ above the plate (`capstone_data.SENSOR_HEIGHT`).

**Physics.** A point source of strength $q$ at $s=(x,y)$ produces at sensor $i$ (position $p_i$) a field magnitude that falls off with the inverse square of the three-dimensional distance:
$$f_i(s,q)=\frac{q}{\lVert s-p_i\rVert^2+h^2}.$$
The height $h$ keeps the magnitude finite even when the source is directly below a sensor.
The **forward operator** $\mathcal F(s,q)=(f_1,\dots,f_8)$ is cheap, exact, and nonlinear in $s$ (but linear in $q$).

**Observation model.** The stored measurements are noisy: each reading is multiplied by $1+0.03\varepsilon_i$ with independent $\varepsilon_i\sim N(0,1)$ (`book1:gaussian-distribution`), then rounded to four decimals.
So $y_i=f_i(s,q)(1+0.03\varepsilon_i)$: a **3% multiplicative** noise.
Taking logs, $\log y_i=\log q-\log(\lVert s-p_i\rVert^2+h^2)+\log(1+0.03\varepsilon_i)$, and the last term is approximately $0.03\varepsilon_i$: additive noise of constant size on the log scale.

**Vectorized forward model.** With sources as an `(N, 2)` tensor, strengths `(N,)`, and sensors `(S, 2)`, broadcasting (`book1:broadcasting`) computes all squared distances at once:
`d2 = ((source[:, None, :] - sensors[None, :, :]) ** 2).sum(-1)` has shape `(N, S)`, and the magnitudes are `strength[:, None] / (d2 + h**2)`.
Practice p10 asks for exactly this as `forward_field(source_xy, strength, sensors_xy)`.

**Checkpoint 1A.** A source of strength $q=1$ sits exactly at the plate's center $(0.5,0.5)$. What magnitude does the corner sensor $(0,0)$ read without noise?

**Checkpoint 1B.** Why does multiplicative noise suggest feeding $\log y$ to a learned model?

**Collected answers.** $\lVert s-p\rVert^2=0.25+0.25=0.5$, so $f=1/(0.5+0.0625)=1/0.5625\approx1.778$. Because on the log scale the noise becomes approximately additive with constant size $0.03$, and the strength becomes an additive offset, which suits a model whose first layer is linear.

## 2. Ill-posedness and noise

A problem is **well-posed** (Hadamard) when a solution exists, is unique, and depends continuously on the data.
Inverse problems often fail the last two conditions.

**Uniqueness.** Three unknowns $(x,y,q)$ from eight readings is not underdetermined, and for sources inside $[0.1,0.9]^2$ the readings do determine the source; but near-symmetric configurations come close to ambiguity.
A source at $(x,y)$ and its mirror image $(1-x,y)$ produce the same eight readings with the left and right sensors swapped; near the midline $x=0.5$ the two sets differ by less than the noise, so the data barely tell which side is which.

**Stability.** The relevant question is how much a small change in the data moves the estimate.
The inverse-square law flattens with distance: for a source far from every sensor, moving it by $0.01$ changes each reading by a fraction of a percent, comparable to the 3% noise.
Errors in the data are then **amplified** into much larger errors in position.
Sources near the center of the plate are the hardest, because they are far from all sensors at once.

**Noise sets a floor.** No method can beat the information in the data: with 3% noise per reading, even the true source produces a relative misfit of about $0.03$ between $\mathcal F(s,q)$ and the observed $y$.
Section 6 uses this **noise floor** to judge a physics residual.

**Regularization** restores stability by preferring "plausible" solutions: a penalty on size or roughness (Section 4), or, implicitly, a learned model trained only on plausible sources (Section 5).

**Checkpoint 2A.** Why are sources near the plate's center harder to locate than sources near a sensor?

**Checkpoint 2B.** With 3% independent multiplicative noise on each of 8 readings, roughly what relative residual $\lVert\mathcal F(s,q)-y\rVert/\lVert y\rVert$ does the *true* source produce?

**Collected answers.** They are far from all sensors, where the inverse-square field is flat, so a small displacement changes the readings by less than the noise; the noise is amplified into a large position error. About $0.03$ (each reading is off by about 3% of itself, so the vector misfit is about 3% of the vector).

## 3. Linearization by discretization: the grid operator

The forward map is nonlinear in the position, but it is **linear in the strength**.
Discretize the plate: fix a $9\times9$ grid of candidate source positions $g_j$ on $[0,1]^2$ (spacing $1/8$; `capstone_data.GRID_XY`, shape `(81, 2)`), and allow a nonnegative "density" $u_j$ at every node.
The readings of a superposition of sources add, so
$$y\approx A u,\qquad A_{ij}=\frac{1}{\lVert g_j-p_i\rVert^2+h^2},$$
where column $j$ of the $8\times81$ matrix $A$ is the field of a **unit** source at node $j$.
A single source at a node is a density with one nonzero entry equal to $q$.

**Heavily underdetermined.** There are 81 unknowns and 8 equations.
`book1:invertibility-via-rank`: $A$ has rank at most 8, so $A^\top A$ ($81\times81$) has rank at most 8 and is **singular**; $Au=y$ has infinitely many solutions whenever it has one, differing by any vector in the 73-dimensional null space of $A$.
Least squares alone cannot choose among them: the normal equations $A^\top A u=A^\top y$ have no unique solution.
This is the motivation for regularization and for learned inversion, and the reason this unit's grid baseline is weak.

**Reading out a source.** From a density $u$ (clipped to $u\ge0$), the baseline takes the three nodes with the largest density, reports their density-weighted centroid as the source position, and their total mass as the strength.

**Checkpoint 3A.** What are the shapes of $A$, $A^\top A$, and $A^\top y$ for one reading vector $y$?

**Checkpoint 3B.** Why can $A^\top A$ not be inverted, and what is the dimension of $A$'s null space if its rank is 8?

**Collected answers.** $A$ is $8\times81$, $A^\top A$ is $81\times81$, $A^\top y$ is $81\times1$ (or `(81,)`). Its rank is at most 8 < 81, so it is singular (`book1:invertibility-via-rank`); the null space has dimension $81-8=73$.

## 4. Tikhonov regularization is ridge regression

**Tikhonov regularization** adds a penalty on the size of $u$:
$$J(u)=\lVert Au-y\rVert^2+\lambda\lVert u\rVert^2,\qquad\lambda>0.$$
This is exactly ridge regression (`book1:l2-regularization`) with design matrix $A$ and coefficients $u$.

**The minimizer.** Using `book1:sum-of-squares-gradients`, $\nabla_u\lVert Au-y\rVert^2=2A^\top(Au-y)$ and $\nabla_u\lambda\lVert u\rVert^2=2\lambda u$.
Setting the gradient (`book1:gradient`) to zero:
$$A^\top A u+\lambda u=A^\top y\quad\Longrightarrow\quad u_\lambda=(A^\top A+\lambda I)^{-1}A^\top y .$$
For $\lambda>0$ the matrix $A^\top A+\lambda I$ is invertible: for any $v\ne0$, $v^\top(A^\top A+\lambda I)v=\lVert Av\rVert^2+\lambda\lVert v\rVert^2>0$, so $(A^\top A+\lambda I)v\ne0$ and the matrix has full rank.
Practice p13 asks you to write this out and prove that $u_\lambda$ is the **unique** minimizer.
In code, solve the linear system rather than forming an inverse: `torch.linalg.solve(A.T @ A + lam * torch.eye(81), A.T @ y)`.

**What $\lambda$ does.** Small $\lambda$: close to the minimum-norm least-squares solution, sensitive to noise. Large $\lambda$: small densities everywhere, stable but biased toward zero.
Choose $\lambda$ on validation rows; this unit's committed baseline fixes $\lambda=1$ (`capstone_data.TIKHONOV_LAMBDA`), and the baseline's error barely changes over several orders of magnitude of $\lambda$.

**Why the baseline is weak.** Among all densities that explain the data, the norm penalty prefers the one with the smallest $\lVert u\rVert$.
Columns of $A$ for nodes **near a sensor** are large (up to $1/h^2=16$), so a small density there explains a lot of signal cheaply.
The regularized solution therefore pushes mass toward the boundary nodes nearest the strongest readings rather than onto the true interior node.
The top-3 centroid then lands between the boundary and the truth.
This is a property of the method, not a bug, and it is why the p19 statement prints the baseline's error ($0.2137$ at the frozen seed) as the number to beat, and why a learned inverse can beat it by a wide margin.

**Checkpoint 4A.** Derive the gradient of $J(u)$ and solve for the stationary point.

**Checkpoint 4B.** Why does Tikhonov's norm penalty favour grid nodes near sensors in this problem?

**Collected answers.** $\nabla J=2A^\top(Au-y)+2\lambda u$; setting it to zero gives $(A^\top A+\lambda I)u=A^\top y$, so $u=(A^\top A+\lambda I)^{-1}A^\top y$ (the matrix is invertible for $\lambda>0$). Columns for nodes near a sensor have large entries, so a small density there produces a large reading; explaining the data with small densities at those nodes gives a smaller $\lVert u\rVert^2$ than putting the true mass on an interior node.

## 5. Learned inversion from simulated pairs

Because the forward operator is known and cheap, we can **simulate** as many (source, readings) pairs as we like and train a network to map readings back to sources.
The `inverse` dataset's 800 training rows are exactly such simulated pairs (with the observation noise), and `capstone_data.simulate("inverse", n, seed)` draws more for the accelerator extension.

**The model.** An MLP (`book1:mlp-architecture`, `book1:relu-activation`) `8 → 64 → 64 → 3` predicting $(x,y,q)$, trained with MSE (`book1:mse-loss`) by full-batch Adam (`book1:torch-optimizers`, `book1:autograd-training`).
Put the preprocessing *inside* the module: `forward(magnitudes)` takes the raw `(B, 8)` readings exactly as `train_rows` returns them, computes `torch.log`, and standardizes with the training-set mean and standard deviation stored as **registered buffers**.
Then the model is self-contained (it can be saved and moved to a GPU as one object), and the seam check can hash the raw rows that reach `forward`.

**Amortization.** Training costs a few seconds once; afterwards each inversion is a single forward pass.
Compare a per-example method (an optimizer run for each new reading vector): a learned inverse **amortizes** the cost of inversion over all future inputs.

**Coverage of the source domain.** The network learns the inverse only where training sources exist: here $[0.1,0.9]^2$ and $q\in[0.5,2]$.
Outside that region it **extrapolates**, and nothing in training constrains it.
A good report states the domain and checks behaviour at and beyond its edges (the laboratory does), and the accelerator extension can widen coverage by simulating more pairs, but only within the domain you are asked to solve.

**Checkpoint 5A.** Why store the log-feature mean and standard deviation as buffers rather than plain attributes?

**Checkpoint 5B.** A learned inverse trained on sources in $[0.1,0.9]^2$ is given readings from a source at $(0.02, 0.5)$. What should you expect, and how would you check it?

**Collected answers.** Buffers move with `.to(device)` and are saved in the `state_dict`, so the model is self-contained and its preprocessing cannot drift; plain attributes would stay on the CPU and be lost on save. Unreliable output (extrapolation outside the training domain); simulate sources just outside the domain with the forward model and measure the error there, compared with inside.

## 6. Physics residuals and the open-ended workflow

A learned inverse can be checked **without true sources**: push its prediction back through the forward operator and compare with what was measured.
The **relative physics residual** of a prediction $(\hat s,\hat q)$ for readings $y$ is
$$\rho=\frac{\lVert\mathcal F(\hat s,\hat q)-y\rVert_2}{\lVert y\rVert_2}.$$
It tests position *and* strength together (the position-error metric ignores strength).
Judge it against the **noise floor**: the residual of the true source, about $0.03$ here.
A mean residual near the floor means the predictions explain the data as well as the truth does; a residual several times the floor flags systematic errors, even on unlabelled field data.

**The open-ended workflow for an inverse problem.**

1. Write the forward operator and test it against stored pairs (the residual of the true sources should sit at the noise floor).
2. Compute the committed baseline under the protocol, and understand why it is weak or strong.
3. Train a learned inverse on training pairs, select on validation (position error, residual), never touching the test rows.
4. Check coverage at the domain edges and robustness to extra sensor noise.
5. Optionally **refine**: start from the learned prediction and run a few steps of per-example least squares on the readings (Session 6 does this for mixtures); this costs optimizer steps, so count them.
6. Score once on the locked test set and write the evaluation section of Session 3.

Practice p19 certifies a learned inverse against the baseline with a residual bound; p26 is the open-ended mini-competition on the same data.

**Checkpoint 6A.** Why can the physics residual be computed for field data with no known source, while the position error cannot?

**Checkpoint 6B.** A model's mean validation residual is $0.12$ while the noise floor is $0.03$. What does that suggest?

**Collected answers.** The residual needs only the prediction, the forward operator, and the measured readings; the position error needs the true position. Systematic misfit: predictions explain the data four times worse than the truth does (wrong strengths or positions), so the model is underfitting or biased somewhere; inspect where the residual is largest.

## 7. Worked laboratory: forward model, grid baseline, learned inverse, residual, and coverage

The laboratory verifies the forward operator against stored validation pairs (the true sources' residual is the noise floor), rebuilds $A$ and checks its rank, reproduces the committed grid-Tikhonov estimates, trains a small learned inverse for 300 steps (validation only, no test call), computes its residual, and finally probes coverage with noise-free simulated sources inside and just outside the training domain.
It also counts how often the Tikhonov density's largest node lies on the grid boundary, which makes the bias of Section 4 visible.

**Checkpoint 7A.** Which printed number is the noise floor, and which model number should be compared with it?

**Checkpoint 7B.** What does the outside-domain error show about learned inversion?

**Collected answers.** The residual of the true validation sources; compare the learned inverse's mean validation residual with it. That the network extrapolates poorly beyond the simulated source domain: its error outside the domain is several times its error inside.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

sensors, h = capstone_data.SENSORS_XY, capstone_data.SENSOR_HEIGHT
val_x, val_y = capstone_data.val_rows("inverse")
train_x, train_y = capstone_data.train_rows("inverse")

def field(source_xy, strength):
    d2 = ((source_xy[:, None, :].double() - sensors[None, :, :]) ** 2).sum(-1)
    return strength.double()[:, None] / (d2 + h ** 2)

def residual(pred, readings):
    return ((field(pred[:, :2], pred[:, 2]) - readings.double()).norm(dim=1) / readings.double().norm(dim=1))

floor = residual(val_y, val_x).mean().item()
print("noise floor (true sources):", round(floor, 4))

grid = capstone_data.GRID_XY
A = 1.0 / (((sensors[:, None, :] - grid[None, :, :]) ** 2).sum(-1) + h ** 2)
print("A shape", tuple(A.shape), "rank", torch.linalg.matrix_rank(A).item(),
      "| rank of A^T A", torch.linalg.matrix_rank(A.T @ A).item(), "of 81")
density = torch.linalg.solve(A.T @ A + capstone_data.TIKHONOV_LAMBDA * torch.eye(81, dtype=torch.float64), A.T @ val_x.double().T).T.clamp_min(0)
top1 = grid[density.argmax(dim=1)]
on_boundary = ((top1 == 0) | (top1 == 1)).any(dim=1).double().mean().item()
print(f"largest-density node lies on the grid boundary for {on_boundary:.0%} of validation rows")
tik = capstone_data.tikhonov_grid_predict(val_x)
print("grid-Tikhonov validation position error:", round(capstone_data.position_error_per_example(tik, val_y).mean().item(), 4))

In [ ]:
class LearnedInverse(nn.Module):
    def __init__(self):
        super().__init__()
        logs = torch.log(train_x)
        self.register_buffer("mu", logs.mean(dim=0))
        self.register_buffer("sd", logs.std(dim=0))
        self.net = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, magnitudes):
        return self.net((torch.log(magnitudes) - self.mu) / self.sd)

torch.manual_seed(SEED)
inverse = LearnedInverse()
optimizer = torch.optim.Adam(inverse.parameters(), lr=3e-3)
for step in range(300):
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(inverse(train_x), train_y)
    loss.backward()
    optimizer.step()
with torch.no_grad():
    pred = inverse(val_x)
print("learned inverse (300 steps): validation position error",
      round(capstone_data.position_error_per_example(pred, val_y).mean().item(), 4),
      "| mean residual", round(residual(pred, val_x).mean().item(), 4))

gen = torch.Generator().manual_seed(SEED + 5)
inside = 0.1 + 0.8 * torch.rand(200, 2, generator=gen, dtype=torch.float64)
outside = torch.stack([0.02 * torch.rand(200, generator=gen, dtype=torch.float64),
                       0.1 + 0.8 * torch.rand(200, generator=gen, dtype=torch.float64)], dim=1)   # x in [0, 0.02]
q = torch.full((200,), 1.0, dtype=torch.float64)
with torch.no_grad():
    for name, src in (("inside", inside), ("outside (x < 0.02)", outside)):
        est = inverse(field(src, q).float())
        err = (est[:, :2].double() - src).norm(dim=1).mean().item()
        print(f"noise-free sources {name}: mean position error {err:.4f}")

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Forgetting the sensor height.* Fix: $r^2+h^2$ in the denominator; test the forward model against stored pairs.
2. *Inverting $A^\top A$.* Fix: it is singular; add $\lambda I$ and use `torch.linalg.solve`.
3. *Dropping $\lambda$ from the system.* Fix: $(A^\top A+\lambda I)u=A^\top y$; check the normal-equation residual.
4. *Raw magnitudes fed to the network.* Fix: log features inside `forward`, statistics from training rows as buffers.
5. *Preprocessing outside the model.* Fix: the model takes raw rows, so the seam check and device moves work.
6. *Judging a residual without the noise floor.* Fix: compute the true sources' residual first.
7. *Claiming accuracy outside the simulated domain.* Fix: state the domain; check coverage at its edges.
8. *Tuning $\lambda$ or the network on test rows.* Fix: validation only; one `final_test_score` call.

**Exam connections.** p10 implements `forward_field`, `grid_operator`, and `tikhonov_solve`; p13 derives the Tikhonov minimizer and proves uniqueness for $\lambda>0$; p19 trains a learned inverse and certifies it against the grid baseline with a physics-residual bound; p21 and p22 use the same data for evaluation and campaign design; p26 is the open-ended inverse-problem mini-competition with a step budget and a written rubric.
Round 2's scientific-ML tasks reward exactly this workflow: a correct forward model, a stated baseline, a validated learned method, and an honest evaluation.

**Checkpoint 8A.** Which pitfall makes `torch.linalg.solve(A.T @ A, A.T @ y)` fail or return garbage?

**Checkpoint 8B.** Which practice asks for a uniqueness proof?

**Collected answers.** Pitfall 2 (and 3): $A^\top A$ is singular (rank at most 8), so the system has no unique solution; add $\lambda I$. p13.

## 9. Forward-only deeper topic: Bayesian inversion, adjoints, and physics-informed networks

Forward-only deeper topic.
**Bayesian inversion** treats the source as a random variable with a prior and reports a posterior distribution (for example by sampling), so uncertainty in the source is explicit; Tikhonov's estimate is the posterior mode under a Gaussian prior and Gaussian noise.
For forward models given by large simulations (partial differential equations), gradients of the misfit with respect to millions of unknowns are computed by **adjoint methods**, the scientific-computing twin of backpropagation.
**Physics-informed neural networks** add the residual of the governing equation to the training loss, so the network is penalized for violating the physics even where no data exist.
None of these are prerequisites, permitted shortcuts, or assessed APIs in B2-024.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Under which prior and noise model is Tikhonov's estimate the posterior mode?

**Checkpoint 9B.** What does a physics-informed loss add to an ordinary data-fit loss?

**Collected answers.** A zero-mean Gaussian prior on the unknowns and Gaussian observation noise (with $\lambda$ the ratio of the noise variance to the prior variance). A penalty on the residual of the governing physical equation, evaluated even at points without data.